[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [GeoPandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)

# Reading and Writing Files &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import pyogrio

COUNTRIES_URL = ("https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/data/nordic-countries.geojson")
local_copy = Path("../../data/nordic-countries.geojson")
countries = gpd.read_file(local_copy if local_copy.exists() else COUNTRIES_URL)


weather_stations = pd.DataFrame({
    "name": ["Bergen", "Oslo", "Svalbard", "Tromso", "Kirkenes"],
    "latitude": [60.39, 59.91, 78.22, 69.65, 69.73],
    "longitude": [5.32, 10.75, 15.65, 18.96, 30.05],
})


def mainland_norway():
    """Norway's outline south of 74 degrees north, without Svalbard, in UTM zone 33N."""
    parts = countries[countries["ADMIN"] == "Norway"].to_crs(25833).explode()
    return parts[parts.geometry.bounds["maxy"] < 8_200_000].union_all()


def station_network(n=300, seed=2026):
    """A made-up register of weather stations on mainland Norway, one row each."""
    rng = np.random.default_rng(seed)
    land = mainland_norway()
    x_min, y_min, x_max, y_max = land.bounds
    east, north = [], []
    while len(east) < n:
        x, y = rng.uniform(x_min, x_max), rng.uniform(y_min, y_max)
        if land.contains(shapely.Point(x, y)):
            east.append(x)
            north.append(y)
    lonlat = gpd.GeoSeries(gpd.points_from_xy(east, north), crs=25833).to_crs(4326)
    return pd.DataFrame({
        "station": [f"NS{i:03d}" for i in range(1, n + 1)],
        "latitude": lonlat.y.round(4).to_numpy(), "longitude": lonlat.x.round(4).to_numpy(),
        "elevation_m": np.clip(rng.gamma(1.6, 220, n), 2, 1900).round(0).astype(int),
        "owner": rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2]),
        "installed": rng.integers(1950, 2025, n),
    })


DISTRICT_CENTERS = {"South": (8.0, 58.6), "East": (10.8, 60.5), "West": (6.0, 60.8), "Mid": (11.0, 63.6),
                    "North": (14.5, 67.0), "Far North": (19.0, 69.4), "Northeast": (25.5, 70.0),
                    "Svalbard": (16.0, 78.5)}


def forecast_districts():
    """Eight made-up forecast districts that cover Norway, in UTM zone 33N."""
    norway = countries[countries["ADMIN"] == "Norway"].to_crs(25833).geometry.iloc[0]
    centers = gpd.GeoSeries(gpd.points_from_xy(*zip(*DISTRICT_CENTERS.values())), crs=4326).to_crs(25833)
    cells = shapely.voronoi_polygons(shapely.MultiPoint(list(centers)),
                                     extend_to=shapely.box(*norway.buffer(200_000).bounds)).geoms
    names = [next(name for name, center in zip(DISTRICT_CENTERS, centers) if cell.contains(center))
             for cell in cells]
    districts = gpd.GeoDataFrame({"district": names}, geometry=[cell.intersection(norway) for cell in cells],
                                 crs=25833)
    return districts.set_index("district").loc[list(DISTRICT_CENTERS)].reset_index()


network = station_network()
stations = gpd.GeoDataFrame(
    network, geometry=gpd.points_from_xy(network["longitude"], network["latitude"]), crs=4326
)


named = gpd.GeoDataFrame(
    weather_stations,
    geometry=gpd.points_from_xy(weather_stations["longitude"], weather_stations["latitude"]),
    crs=4326,
)


districts = forecast_districts()
scratch = Path("scratch")
scratch.mkdir(exist_ok=True)
print("geopandas", gpd.__version__, "| pyogrio", pyogrio.__version__, "| GDAL", pyogrio.__gdal_version_string__,
      "|", len(stations), "made-up stations,", len(districts), "made-up districts")

geopandas 1.2.0 | pyogrio 0.13.0 | GDAL 3.12.4 | 300 made-up stations, 8 made-up districts


The worked examples' GeoPackage, rebuilt here so that each answer runs on its own. The stations and districts are made up.

In [2]:
atlas = scratch / "atlas.gpkg"
districts.to_file(atlas, layer="districts")
stations.to_file(atlas, layer="stations")
print(pyogrio.list_layers(atlas)[:, 0])

['districts' 'stations']


**1.** Write `districts` to a GeoJSON file in `scratch`, read it back, and print its CRS and the first four lines of the file.

In [3]:
districts.to_file(scratch / "districts.geojson")
print(gpd.read_file(scratch / "districts.geojson").crs)
for line in (scratch / "districts.geojson").read_text().splitlines()[:4]:
    print(line)

EPSG:25833
{
"type": "FeatureCollection",
"name": "districts",
"crs": { "type": "name", "properties": { "name": "urn:ogc:def:crs:EPSG::25833" } },


The districts are in UTM zone 33N, and GDAL writes that CRS into the file's `crs` member, so the file comes back in EPSG:25833. A program that follows GeoJSON's standard strictly expects longitude and latitude, so convert with `to_crs(4326)` before sending a GeoJSON file to someone else.

**2.** Add the five named stations to `atlas.gpkg` as a layer named `named`, then list the file's layers.

In [4]:
named.to_file(atlas, layer="named")
pyogrio.list_layers(atlas)

array([['districts', 'MultiPolygon'],
       ['stations', 'Point'],
       ['named', 'Point']], dtype=object)

A new layer name adds a layer, and the two layers already in the file stay as they were.

**3.** Read only the stations owned by `road` from the `stations` layer of `atlas.gpkg`, keeping the `station` and `elevation_m` columns, and print how many there are and their mean elevation.

In [5]:
road = gpd.read_file(atlas, layer="stations", where="owner = 'road'",
                     columns=["station", "elevation_m"])
print(len(road), "road stations | mean elevation:", round(road["elevation_m"].mean()), "m")
print("in the register:", (stations["owner"] == "road").sum())

99 road stations | mean elevation: 363 m
in the register: 99


The condition is SQL, so the text `road` goes in single quotes inside the Python string.

**4.** Read the stations inside the Mid district with `mask`, and check the count against a spatial join.

In [6]:
mid = districts[districts["district"] == "Mid"]
in_mid = gpd.read_file(atlas, layer="stations", mask=mid)
print(len(in_mid), "by mask |", len(gpd.sjoin(stations.to_crs(districts.crs), mid)), "by a spatial join")

69 by mask | 69 by a spatial join


`mask` converts the Mid district from meters to the layer's degrees before it filters, so the two counts agree.

**5.** Write the stations to a Shapefile with `elevation_m` renamed so that no column name is cut short, read it back, and print its columns.

In [7]:
stations.rename(columns={"elevation_m": "elev_m"}).to_file(scratch / "short_names.shp")
print(list(gpd.read_file(scratch / "short_names.shp").columns))

['station', 'latitude', 'longitude', 'elev_m', 'owner', 'installed', 'geometry']


Every name is now ten characters or fewer, so GDAL writes them unchanged and warns about nothing.

The scratch folder and its files can go now.

In [8]:
shutil.rmtree(scratch)
print("scratch folder still there:", scratch.exists())

scratch folder still there: False


---

&#8592; **Back to:** [Reading and Writing Files](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/geopandas-deep-dive/05-reading-and-writing-files.ipynb)  &nbsp;&middot;&nbsp;  [GeoPandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)
